# Bag-of-Words MaxRL

Sample script for zero-step-sampling MaxRL on the bag-of-words dataset.

The policy is a Gaussian around the model's scalar prediction, m_theta(z | x) = Normal(f_theta(x), sigma^2). MaxRL weights use the Gaussian likelihood of the noisy target under each rollout.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.maxrl import BagOfWordsMaxRLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/maxrl.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsMaxRLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-maxrl-example-prime",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    subtract_baseline=True,
    num_rollouts_per_sample=1024,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Canonical degree: {config.degree}")
print(f"Policy stdev:     {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-maxrl-example-prime/7-words_corr-0.2_len-128_pow-1.0_ar-0.5
Dataset corr target: 0.2000
Backbone lr: 1.230e-03
Head lr:     8.192e-03
Rollouts/sample: 1024
Canonical degree: 1024
Policy stdev:     1.0


Max token length (train): 128
Max token length (val):   128
ceil_padded_seqlen:       128


Tokenize train:   0%|          | 0/49 [00:00<?, ?it/s]

Tokenize val:   0%|          | 0/49 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
state.run_training()

maxrl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

maxrl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
analysis = BagOfWordsAnalysisConfig.from_studies({"example": config.study_folder})
epoch_axis = pl.col("epoch").alias("epoch")

analysis.xy_plots([
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, rsq_expr(split="val", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="val", y="ground_truth"), None),
])

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()